# 08 · The saved models: re-load each one and re-run it

There are 21 models: LR, XGBoost and ResNet-50, each image only and + morphometrics, plus
the morphometric-only MLP, on the three rat tasks.

This notebook loads each model from disk with its own weights and normalisation statistics,
re-scores the rat test split through the package's evaluation code, and checks that the
predicted labels and probabilities are the ones saved at training time. `HUMAN = True` also
re-scores the human comparisons with the two ResNet-50s. Outputs go to
`runs/verify_models/` (`runs/synthetic/verify_models/` on synthetic data).

* **Synthetic data:** checks the models notebooks 02–05 trained (run those first). This is
  the default.
* **Original data and weights:** checks the 21 models behind the manuscript (about 15 min
  with a GPU). The weights are not distributed.

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
GO = SYNTHETIC      # <- True to re-run the saved models
HUMAN = True        # also the human comparisons
if GO:
    cmd = [sys.executable, "-u", str(HERE / "scripts" / "verify_models.py")] + (["--human"] if HUMAN else [])
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)
v = P.WORK / "verify_models" / "verification.csv"
pd.read_csv(v) if v.is_file() else print("not run yet")

### Using a saved model on new patches

The same machinery scores any folder of patches laid out as `<folder>/<class index>/*.png`
(the class index is only used for scoring; use `0` if unknown). Slide-level calls are the
patch probabilities averaged per slide, where a slide id is the file name up to `__r`.

In [ ]:
from verify_models import config
from vmlevaluator.evaluate.runner import evaluate_experiment
cfg = config("control_vml", "cnn_morph")                  # Control vs VML, ResNet-50 + morph
cfg.data.test.img_path = str(P.HUMAN_ROOT / "uninjured_1mo")  # <- any folder of patches
cfg.data.class_names = ["Uninjured", "1-month post-ACLR"]
cfg.data.morphometrics_path = str(P.HUMAN_WB)               # <- workbook with the slides' mean CSA and fibre count
if GO:
    r = evaluate_experiment(cfg, output_dir=str(P.WORK / "example_prediction"), make_plots=False, verbose=False)
    print(r)
    display(pd.read_csv(P.WORK / "example_prediction" / "wsi_predictions.csv").head())